In [24]:
import ollama

In [25]:
inventory_db = {
    "apple": {"price": 1.0, "quantity": 100},
    "banana": {"price": 0.5, "quantity": 150},
    "orange": {"price": 0.8, "quantity": 120}
}

In [26]:
# Tool 1: Check inventory
def check_inventory(item):
    item_name = item.lower()
    if item_name in inventory_db:
        return f"{item.capitalize()} - Price: ${inventory_db[item_name]['price']}, Quantity: {inventory_db[item_name]['quantity']}"
    else:
        return f"{item.capitalize()} is not available in the inventory."
    
# Tool 2: Business Logic for discounts for loyal customers
def calculate_loyality_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.25)  # Max discount of 25%
    discounted_price = base_price * (1 - discount)
    return discounted_price


In [27]:
available_tools = {
    "check_inventory": check_inventory,
    "calculate_loyality_discount": calculate_loyality_discount
}

In [28]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check the price and quantity of an item in the inventory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {"type": "string", "description": "Name of the item to check in inventory."}
                },
                "required": ["item"]
            }
        }
    }, 
    {
        "type": "function",
        "function": {
            "name": "calculate_loyality_discount",
            "description": "Calculate the discount for loyal customers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "float", "description": "The original price of the item."},
                    "years_as_customer": {"type": "integer", "description": "Number of years the customer has been with the business."}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [29]:
messages = [
    {"role": "user", "content": "I wants to buy the apple, how much it cost?"},
]

response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Model Response:", response)

Model Response: model='llama3.1:8b' created_at='2026-05-24T15:58:27.3968864Z' done=True done_reason='stop' total_duration=54253376300 load_duration=10038529900 prompt_eval_count=249 prompt_eval_duration=40138209600 eval_count=17 eval_duration=4048296200 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'item': 'apple'}))]) logprobs=None


In [30]:
message = response["message"]
print("Model Message:", message)

Model Message: role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'item': 'apple'}))]


In [36]:
tool_calls = response["message"].get("tool_calls", [])
if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]
        
        if tool_name in available_tools:
            result = available_tools[tool_name](**tool_args)
            print(f"Tool '{tool_name}' called with arguments {tool_args} returned: {result}")

            messages.append(message)
            messages.append({"role": "tool", "content": str(result)})
        else:
            print(f"Tool '{tool_name}' is not available.")



Tool 'calculate_loyality_discount' called with arguments {'base_price': 1, 'years_as_customer': 5} returned: 0.75


In [37]:
final_response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Final Model Response:", final_response)

Final Model Response: model='llama3.1:8b' created_at='2026-05-24T16:05:46.9064196Z' done=True done_reason='stop' total_duration=36151698200 load_duration=3604456300 prompt_eval_count=162 prompt_eval_duration=26508341200 eval_count=25 eval_duration=6007138100 message=Message(role='assistant', content='{"name": "calculate_discount", "parameters": {"years": "5", "discount_per_year": "0"}}', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [38]:
print(final_response["message"]["content"])

{"name": "calculate_discount", "parameters": {"years": "5", "discount_per_year": "0"}}


In [39]:
prompt = "I am a loyal customer for 5 years, how much discount I can get on the apple?"

In [40]:
messages.append({"role": "user", "content": prompt})
response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Response message:", response["message"])


Response message: role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='calculate_loyality_discount', arguments={'years_as_customer': 5, 'base_price': 1}))]


In [41]:
tool_calls = response["message"].get("tool_calls", [])
if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]
        
        if tool_name in available_tools:
            result = available_tools[tool_name](**tool_args)
            print(f"Tool '{tool_name}' called with arguments {tool_args} returned: {result}")

            messages.append(message)
            messages.append({"role": "tool", "content": str(result)})
        else:
            print(f"Tool '{tool_name}' is not available.")


Tool 'calculate_loyality_discount' called with arguments {'years_as_customer': 5, 'base_price': 1} returned: 0.75


In [42]:
final_response_with_discount = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Final Model Response with Discount:", final_response_with_discount["message"])

Final Model Response with Discount: role='assistant' content='Since you are a loyal customer for 5 years, and previously the discount was calculated as $1.0 * 0.75 = $0.75, we can further apply an additional loyalty discount of 10% on top of the previous discount.\n\nThe new total discount would be: $0.75 * 0.90 (10% off) = $0.675 \n\nYou will get a total of 67.5% discount on the apple price.' thinking=None images=None tool_name=None tool_calls=None
